# Simulation Model Fitting

Create `multidms.Data` objects and fit models across a grid of
fusion regularization values for each library and measurement type.

In [ ]:
config_path = "config/config.yaml"
profile = None

In [ ]:
import warnings
import os
import pickle

import pandas as pd
import numpy as np
import multidms

from notebooks._common import load_config

In [ ]:
# Load configuration
cfg = load_config(config_path, profile)
sim = cfg["simulation"]
fit = sim["fitting"]

seed = cfg["seed"]
output_dir = sim["output_dir"]
gpu_ids = cfg["gpu_ids"]
n_processes = cfg["n_processes"]

warnings.filterwarnings("ignore", category=FutureWarning)warnings.filterwarnings("ignore", category=DeprecationWarning)

## Load simulated functional scores

In [ ]:
func_scores = pd.read_csv(f"{output_dir}/simulated_func_scores.csv")
func_scores["func_score_type"] = pd.Categorical(
    func_scores["func_score_type"],
    categories=["observed_phenotype", "loose_bottle", "tight_bottle"],
    ordered=True,
)
print(f"Loaded {len(func_scores)} rows")

## Create Data objects

One `multidms.Data` object per (library, measurement_type) combination.

In [ ]:
data_objects = []
for (lib, target), fs_df in func_scores.rename(columns={"homolog": "condition"}).groupby(
    ["library", "func_score_type"]
):
    df = fs_df.copy()
    df["aa_substitutions"] = df["aa_substitutions"].fillna("")
    data_objects.append(
        multidms.Data(
            df,
            reference="h1",
            alphabet=multidms.AAS_WITHSTOP_WITHGAP,
            verbose=False,
            name=f"{lib}_{target}_func_score",
        )
    )

print(f"Created {len(data_objects)} Data objects:")
for d in data_objects:
    print(f"  {d.name}")

## Configure fitting parameters

In [ ]:
fitting_params = {
    "maxiter": [fit["maxiter"]],
    "tol": [fit["tol"]],
    "fusionreg": fit["fusionreg_values"],
    "l2reg": [fit["l2reg"]],
    "beta0_ridge": [fit["beta0_ridge"]],
    "ge_type": [fit["ge_type"]],
    "ge_kwargs": [fit["ge_kwargs"]],
    "cal_kwargs": [fit["cal_kwargs"]],
    "loss_kwargs": [fit["loss_kwargs"]],
    "warmstart": [fit["warmstart"]],
    "beta0_init": [fit["beta0_init"]],
    "alpha_init": [fit["alpha_init"]],
    "beta_clip_range": [tuple(fit["beta_clip_range"])],
    "dataset": data_objects,
}
print("Fitting parameters:")
for k, v in fitting_params.items():
    if k != "dataset":
        print(f"  {k}: {v}")

## Fit models

In [ ]:
_, _, fit_collection_df = multidms.model_collection.fit_models(
    fitting_params, gpu_ids=gpu_ids, n_processes=n_processes
)

# Convert dict-valued columns to strings for groupby compatibility
for col in fit_collection_df.columns:
    if fit_collection_df[col].apply(lambda x: isinstance(x, dict)).any():
        fit_collection_df[col] = fit_collection_df[col].apply(str)

print(f"Fit {len(fit_collection_df)} models")

## Post-process and save

In [ ]:
fit_collection_df = fit_collection_df.assign(
    measurement_type=fit_collection_df["dataset_name"].str.split("_").str[2:4].str.join("_"),
    library=fit_collection_df["dataset_name"].str.split("_").str[0:2].str.join("_"),
)

fit_collection_df["measurement_type"] = pd.Categorical(
    fit_collection_df["measurement_type"],
    categories=["observed_phenotype", "loose_bottle", "tight_bottle"],
    ordered=True,
)

print(f"Post-processed {len(fit_collection_df)} models")

In [ ]:
pickle.dump(fit_collection_df, open(f"{output_dir}/fit_collection.pkl", "wb"))
print(f"Saved fit_collection.pkl ({len(fit_collection_df)} models)")